# Data Analytics Lab Sheet 9: Data Storytelling and Business Insight Generation

**Name:** ANURAG PARMAR  
**Roll No.:** 8  
**Section:** B  
**CU ID:** CU24250076  
**Program:** B.Tech CSE  

## Aim
Create visualizations, KPIs, and actionable business recommendations from a reproducible Superstore-style dataset.

> Run cells from top to bottom. Dataset sources and any synthetic-data notes are documented in the relevant sections.

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

## 2. Create a Superstore-style dataset

In [ ]:
rng = np.random.default_rng(42)
n = 800
regions = ["North", "South", "East", "West"]
categories = ["Technology", "Furniture", "Office Supplies"]
segments = ["Consumer", "Corporate", "Home Office"]
months = pd.date_range("2025-01-01", periods=12, freq="MS")
df = pd.DataFrame({
    "Order_ID": np.arange(10001, 10001 + n),
    "Order_Date": rng.choice(months, n),
    "Region": rng.choice(regions, n),
    "Category": rng.choice(categories, n, p=[0.35, 0.30, 0.35]),
    "Segment": rng.choice(segments, n),
    "Sales": np.round(rng.gamma(4.0, 120.0, n), 2),
    "Quantity": rng.integers(1, 10, n),
    "Discount": np.round(rng.choice([0, .05, .10, .15, .20, .30], n), 2)
})
base_margin = df["Category"].map({"Technology": .24, "Furniture": .12, "Office Supplies": .18})
df["Profit"] = np.round(df["Sales"] * (base_margin - df["Discount"] * .45) + rng.normal(0, 18, n), 2)
df["Month"] = df["Order_Date"].dt.strftime("%b")
display(df.head())

## 3. Business objective and KPIs

In [ ]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
total_orders = df["Order_ID"].nunique()
average_order_value = total_sales / total_orders
profit_margin = total_profit / total_sales * 100
kpis = pd.DataFrame({
    "KPI": ["Total Sales", "Total Profit", "Orders", "Average Order Value", "Profit Margin (%)"],
    "Value": [total_sales, total_profit, total_orders, average_order_value, profit_margin]
})
display(kpis.round(2))

## 4. Visualize the business story

In [ ]:
monthly = df.groupby("Order_Date", as_index=False).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
monthly = monthly.sort_values("Order_Date")
monthly.plot(x="Order_Date", y=["Sales", "Profit"], marker="o", figsize=(11, 5))
plt.title("Monthly Sales and Profit")
plt.tight_layout()
plt.show()

region_summary = df.groupby("Region", as_index=False).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
category_summary = df.groupby("Category", as_index=False).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
display(region_summary.sort_values("Sales", ascending=False))
display(category_summary.sort_values("Profit", ascending=False))

sns.barplot(data=region_summary, x="Region", y="Sales")
plt.title("Sales by Region")
plt.show()
sns.barplot(data=category_summary, x="Category", y="Profit")
plt.title("Profit by Category")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

sns.scatterplot(data=df, x="Discount", y="Profit", hue="Category")
plt.title("Discount vs Profit")
plt.show()

## 5. Generate insights and recommendations

In [ ]:
best_region = region_summary.loc[region_summary["Sales"].idxmax(), "Region"]
best_category = category_summary.loc[category_summary["Profit"].idxmax(), "Category"]
loss_orders = int((df["Profit"] < 0).sum())
print(f"Highest-sales region in this sample: {best_region}")
print(f"Highest-profit category in this sample: {best_category}")
print(f"Orders with negative profit: {loss_orders} of {len(df)}")
print("\nRecommendations:")
print("1. Investigate negative-profit orders and discount levels.")
print("2. Compare sales and profit together before prioritizing a region or category.")
print("3. Review monthly trends and validate them against additional historical data.")
print("Note: the dataset is synthetic; insights illustrate a workflow, not real company performance.")
df.to_csv("data_storytelling_sales.csv", index=False)
print("Saved data_storytelling_sales.csv")